# Reasonings of the two-player games
What the models think about while they play (reasoning_analysis.py): the tables of the self-play experiments, from the `.thoughts.jsonl` next to every duel record. Re-run it after copying new results; nothing is computed by hand.
The counts are keyword patterns, so they are rough: they compare prompts, modes and games, they do not judge a single sentence. For that, the last section exports every guess next to the opponent's own reasoning, to label by hand.
Run it locally: copy the results folder from Google Drive (`MyDrive/LLMs-Play-Textual-Games/Benchmark/results`) into `Code/Benchmark/results` or `Code/Benchmark/results_new`.

In [1]:
import os
import sys

import pandas as pd

sys.path.insert(0, os.getcwd())
from records import LEGACY_RESULTS
from reasoning_analysis import GAME_KEY, guess_pairs, outcome_table, reasoning_table, summary

pd.set_option("display.max_colwidth", 200)
folders = [folder for folder in [LEGACY_RESULTS, "results_new"] if os.path.isdir(folder)]
table = reasoning_table(*folders)
print(f"{len(table)} reasonings in {len(table.drop_duplicates(GAME_KEY))} games")

860 reasonings in 65 games


## The comparison table
One row per game and prompt. Columns:
- `games`, `reasonings`: how many; `words_median`: the length of a reasoning;
- `user_frame_%`: reasonings whose first lines call the player "the user" ("Okay, let's see. The user is playing as Black"): the model reasons as an assistant advising someone, not as a player;
- `board_sentences_%`: sentences spent reading the board ("row 3", "rank", "Wait", "let me check"); `waits_median`: "Wait" per reasoning;
- `with_guess_%`, `guesses_per_reasoning`: sentences that attribute an intention to the other player ("O is trying to...", "the opponent might be setting up...", "they are aiming..."): the behaviour the experiment looks for;
- `opponent_word_per_reasoning`: how often the word "opponent" is used (neutral, free and plain never say it).

In [2]:
summary(table)

games  reasonings  words_median  user_frame_%  \
game         prompt                                                   
chess        free        17         295        1317.0         27.12   
             neutral      8         167         569.0         91.02   
             plain        8         198        1259.5         45.45   
connect_four CoT          8          49        2600.0         73.47   
             free         8          38        2067.5         34.21   
             neutral      8          47        2372.0         91.49   
             plain        8          66        2243.0         56.06   

                      board_sentences_%  waits_median  with_guess_%  \
game         prompt                                                   
chess        free                 20.09          13.0          3.39   
             neutral              17.03           3.0          6.59   
             plain                19.39          12.0          5.05   
connect_four CoT                  35.42          22.0         26.53   
             free                 37.40          14.0         42.11   
             neutral              35.82          20.0         27.66   
             plain                33.92          16.0         45.45   

                      guesses_per_reasoning  opponent_word_per_reasoning  
game         prompt                                                       
chess        free                      0.04                         0.02  
             neutral                   0.07                         0.00  
             plain                     0.09                         0.01  
connect_four CoT                       0.76                         1.31  
             free                      1.11                         1.74  
             neutral                   0.45                         0.17  
             plain                     1.18                         0.33

## Per thinking mode
The same, split by the mode of the player who wrote the reasoning (in self-play the agent and the opponent have their own mode). Small groups (few reasonings) are noisy: r4 players think about every 4 moves.

In [3]:
summary(table, by=["game", "prompt", "player_mode"])

games  reasonings  words_median  \
game         prompt  player_mode                                    
chess        free    e1               8         198        1537.0   
                     er4              7          27         938.0   
                     lr4              7          33        1352.0   
                     r4               8          37         748.0   
             neutral e1               5         115         600.0   
                     er4              2          14         725.5   
                     lr4              2           7         544.0   
                     r4               5          31         295.0   
             plain   e1               5         146        1654.0   
                     er4              2          14        1136.0   
                     lr4              2           4        1013.0   
                     r4               5          34         783.5   
connect_four CoT     e1               6          42        2668.5   
                     r4               3           7        2121.0   
             free    e1               6          33        1830.0   
                     r4               3           5        3164.0   
             neutral e1               6          39        2428.0   
                     r4               4           8        1779.5   
             plain   e1               6          53        2732.0   
                     r4               5          13        1944.0   

                                  user_frame_%  board_sentences_%  \
game         prompt  player_mode                                    
chess        free    e1                  19.70              20.66   
                     er4                 25.93              19.83   
                     lr4                 24.24              18.31   
                     r4                  70.27              17.06   
             neutral e1                  93.04              17.29   
                     er4                100.00              15.78   
                     lr4                100.00              10.27   
                     r4                  77.42              18.22   
             plain   e1                  43.15              19.36   
                     er4                 50.00              19.29   
                     lr4                 25.00              11.46   
                     r4                  55.88              20.56   
connect_four CoT     e1                  83.33              34.92   
                     r4                  14.29              38.41   
             free    e1                  39.39              36.23   
                     r4                   0.00              42.27   
             neutral e1                  94.87              35.18   
                     r4                  75.00              40.73   
             plain   e1                  54.72              33.59   
                     r4                  61.54              35.62   

                                  waits_median  with_guess_%  \
game         prompt  player_mode                               
chess        free    e1                   16.0          3.03   
                     er4                   9.0          7.41   
                     lr4                  16.0          6.06   
                     r4                    3.0          0.00   
             neutral e1                    3.0          7.83   
                     er4                   3.5          0.00   
                     lr4                   2.0         14.29   
                     r4                    0.0          3.23   
             plain   e1                   15.5          5.48   
                     er4                  14.5         14.29   
                     lr4                   5.0          0.00   
                     r4                    2.0          0.00   
connect_four CoT     e1                   21.5         30.95   
                     r4             

## Who wins
The games with reasonings: results of the agent (after Stockfish's adjudication in chess, when there is one), how the games ended, illegal answers per game (both players).

In [4]:
outcome_table(*folders)

games  wins  draws  losses  illegal_per_game  \
game         prompt                                                  
chess        free        17     0     17       0              2.94   
             neutral      8     0      8       0              2.50   
             plain        8     0      8       0              2.12   
connect_four CoT          8     4      0       4              0.38   
             free         8     2      0       6              0.00   
             neutral      8     3      0       5              0.25   
             plain        8     5      0       3              0.12   

                                      endings  
game         prompt                            
chess        free     threefold_repetition 17  
             neutral   threefold_repetition 8  
             plain     threefold_repetition 8  
connect_four CoT              four_in_a_row 8  
             free             four_in_a_row 8  
             neutral          four_in_a_row 8  
             plain            four_in_a_row 8

## Examples of guesses
A random sample of the guesses of one game and prompt, with who wrote them, to see what the numbers count.

In [5]:
game, prompt, n = "connect_four", "plain", 15

rows = table[(table.game == game) & (table.prompt == prompt) & (table.guesses.str.len() > 0)]
examples = rows.explode("guesses").sample(min(n, rows.guesses.str.len().sum()), random_state=0)
examples[["mode", "level", "seed", "ply", "color", "player_mode", "move", "guesses"]].sort_values(["mode", "level", "seed", "ply"])

,mode,level,seed,ply,color,player_mode,move,guesses
695,e1,self-e1,1,4,O,e1,3,"For example, if X is trying to make a vertical line in column 3, but there's already an X in row 6."
696,e1,self-e1,1,5,X,e1,4,"But since the opponent is O, and they just played in column 3, maybe they are trying to block me."
697,e1,self-e1,1,6,O,e1,2,So X is trying to make a line.
702,e1,self-e1,1,11,X,e1,6,"For example, if O is trying to get four in a row in some way, but I don't see it."
702,e1,self-e1,1,11,X,e1,6,"For example, if O is trying to connect in some way, X should block."
704,e1,self-e1,1,13,X,e1,5,"But since it's O's turn, they might be trying to do that."
704,e1,self-e1,1,13,X,e1,5,So maybe O is trying to connect in some other way.
719,e1,self-r4,1,6,O,r4,5,So X is trying to build a vertical line in column 1.
720,e1,self-r4,1,7,X,e1,4,"But since O just placed in column 5, maybe they are trying to create a line."
721,e1,self-r4,1,9,X,e1,2,Maybe O is trying to create a line.


## Are the guesses right? Labelling by hand
Every reasoning with a guess, next to the end of the opponent's reasoning on the move before (the move the guess is about, where the opponent decides). `checkable` is False when the opponent did not reason on that move (e.g. an r4 opponent), so the guess cannot be checked against its intention.
The pairs go to `labels/guess_pairs.csv`: fill the column `label` (right, partly, wrong, unclear) in a spreadsheet. Judge the main guess, not any of the hypotheses listed (the model often lists vertical, horizontal and diagonal plans at once). Running this cell again adds the new pairs and keeps the labels already written.

In [6]:
labels_file = "labels/guess_pairs.csv"
pairs = guess_pairs(table)
pair_key = GAME_KEY + ["ply"]
if os.path.exists(labels_file):
    old = pd.read_csv(labels_file, dtype={"level": str, "label": str}).fillna({"label": ""})
    pairs["level"] = pairs.level.astype(str)
    pairs = pairs.drop(columns="label").merge(old[pair_key + ["label"]], on=pair_key, how="left").fillna({"label": ""})
os.makedirs(os.path.dirname(labels_file), exist_ok=True)
pairs.to_csv(labels_file, index=False)
print(f"{len(pairs)} reasonings with guesses, {pairs.checkable.sum()} checkable, "
      f"{(pairs.label != '').sum()} labelled -> {labels_file}")
pairs.groupby(["game", "prompt"]).agg(with_guesses=("ply", "size"), checkable=("checkable", "sum"))

103 reasonings with guesses, 56 checkable, 0 labelled -> labels/guess_pairs.csv


with_guesses  checkable
game         prompt                          
chess        free               10          4
             neutral            11          7
             plain              10          7
connect_four CoT                13          7
             free               16          8
             neutral            13          7
             plain              30         16

## Accuracy of the labelled guesses
Share of right / partly / wrong / unclear among the labelled checkable pairs, per game and prompt.

In [7]:
labelled = pairs[pairs.checkable & (pairs.label != "")]
if labelled.empty:
    print("no labels yet: fill the column label of", labels_file)
else:
    counts = labelled.groupby(["game", "prompt"]).label.value_counts().unstack(fill_value=0)
    display(counts.assign(labelled=counts.sum(axis=1), right_share=(counts.get("right", 0) / counts.sum(axis=1)).round(2)))

no labels yet: fill the column label of labels/guess_pairs.csv
